# Model-3 (CNN - Breath-by-breath data)

Use the Python 3.11 kernel and private preprocessed data; see [setup](../README.md). Run cells top to bottom.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from ai_cpet_ots.dataloaders import Model_3_Dataset, create_dataset_model_3
from ai_cpet_ots.helpers import collate, segment_BBB_data
from ai_cpet_ots.models import Model_3
from ai_cpet_ots.paths import BBB_CONTROL_DIR, BBB_OTS_DIR, MODEL_WEIGHTS, output_path
from ai_cpet_ots.testers import test_model_3

In [ ]:
def optimize_model(X_train, Y_train):
    """Search the notebook's study hyperparameter bounds using training inputs only.

    Returns the optimizer result used by train_model. Requires the optimization
    extra and can be expensive; it is not run in the default evaluation workflow."""
    from ai_cpet_ots.optimizers import Optimizer_Model_3

    opt = Optimizer_Model_3(X_train, Y_train)

    bounds = [
        {"name": "n_epochs", "type": "discrete", "domain": np.arange(5, 55, 5)},
        {"name": "dropout", "type": "continuous", "domain": (0, 0.9)},
        {"name": "lr", "type": "continuous", "domain": (1e-6, 1e-1)},
        {"name": "weight_decay", "type": "continuous", "domain": (0, 1e-2)},
        {"name": "batch_size", "type": "discrete", "domain": (64, 128, 256, 512)},
    ]

    _, opt_params_gpbo = opt.run_bo_gp(bounds, 50)

    return opt_params_gpbo

In [ ]:
def train_model(opt_params, X_train, Y_train):
    """Train this notebook's model from opt_params and the supplied training arrays.

    Keeps the original objective, optimizer, and epoch settings. Returns the
    trained model; checkpoint saving is controlled separately by save_model."""
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    n_epochs, dropout, lr, weight_decay, batch_size, threshold = opt_params

    trainset = Model_3_Dataset(X_train, Y_train)
    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, collate_fn=collate)

    model = Model_3(dropout, threshold).to(device)

    pos_samples = Y_train.sum()
    neg_samples = len(Y_train) - pos_samples
    pos_weight = torch.tensor([neg_samples / pos_samples]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for e in range(n_epochs):
        train_loss = 0

        for batch in trainloader:
            x_batch, y_batch, _ = batch
            x_batch, y_batch = x_batch.float().to(device), torch.tensor(y_batch).to(device)

            y_pred, _, _, _ = model(x_batch)

            loss = criterion(y_pred.flatten(), y_batch.float())
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss.item()

        print(f"Epoch [{e + 1}/{n_epochs}] - train_loss {train_loss / len(trainloader):.4f}")

    return model

In [ ]:
def test_model(X_test, Y_test, model=None):
    """Load the published checkpoint when model is None and evaluate test inputs.

    Constructs the study DataLoader and forwards plot options to the reusable
    evaluator. Private inputs must use the original feature order and scaling."""
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else:
        device = torch.device("cpu")

    if model is None:
        model = Model_3().to(device)
        model.load_state_dict(
            torch.load(MODEL_WEIGHTS / "Model_3.pth", map_location=device, weights_only=True)
        )

    testset = Model_3_Dataset(X_test, Y_test)
    testloader = DataLoader(testset, batch_size=len(testset), collate_fn=collate)
    auroc, auprc, f1, acc = test_model_3(
        model, "Model-3", testloader, 30, 15, True, True, False, True
    )
    print(
        f"AUROC: {auroc:.3f},AUCPR: {auprc:.3f}, F1: {f1:.3f}, ACC: {acc:.3f}, Threshold {model.threshold:.3f}"
    )

In [ ]:
training_model = False
save_model = False
model_name = "Model_3"
window_size = 30
window_overlap = 15

In [ ]:
"""Load breath-by-breath data"""
_, _, X_train, Y_train, X_test, Y_test, _, _ = create_dataset_model_3(
    BBB_CONTROL_DIR, BBB_OTS_DIR, mode="minmax"
)

In [ ]:
if training_model:
    X_train_splitted, Y_train_splitted, seq_lengths = segment_BBB_data(
        X_train, Y_train, window_size=window_size, overlap=window_overlap
    )

    # Optimize model
    print("Optimizing model...")
    opt_params = optimize_model(X_train_splitted, Y_train_splitted)

    # Train the model
    print("Training model...")
    model = train_model(opt_params, X_train_splitted, Y_train_splitted)

    # Test model
    print("Testing model...")
    test_model(X_test, Y_test, model)

    if save_model:
        torch.save(model.state_dict(), output_path(f"{model_name}.pth"))
else:
    print("Testing model...")
    test_model(X_test, Y_test)